In [0]:
from pyspark.sql.functions import col

In [0]:
# Crear esquema silver si no existe
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.m_ubigeo_demografia (
    id_ubigeo STRING COMMENT 'Código único de distrito según INEI',
    nom_distrito STRING COMMENT 'Nombre del distrito',
    nom_provincia STRING COMMENT 'Nombre de la provincia',
    nom_departamento STRING COMMENT 'Nombre del departamento',
    capital_legal STRING COMMENT 'Capital legal del distrito',
    categoria STRING COMMENT 'Clasificación del distrito (Ciudad, Villa, Pueblo)',
    altitud DOUBLE COMMENT 'Altitud promedio del distrito en metros sobre el nivel del mar',
    poblacion INT COMMENT 'Población total del distrito',
    latitud DOUBLE COMMENT 'Latitud geográfica del distrito',
    longitud DOUBLE COMMENT 'Longitud geográfica del distrito'
) USING DELTA;

In [0]:
# Leer Bronze
df_bronze_1 = spark.table("workspace.bronze.data_ubigeo_1")

# Transformar a Silver
df_silver_1 = (
    df_bronze_1
    .withColumnRenamed("ubigeo_Inei", "id_ubigeo")
    .withColumnRenamed("distrito", "nom_distrito")
    .withColumnRenamed("provincia", "nom_provincia")
    .withColumnRenamed("departamento", "nom_departamento")
    .withColumn("altitud", col("altitud").cast("double"))
    .withColumn("poblacion", col("poblacion").cast("int"))
    .withColumn("latitud", col("latitud").cast("double"))
    .withColumn("longitud", col("longitud").cast("double"))
)


In [0]:
# Guardar en tabla Silver
df_silver_1.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.m_ubigeo_demografia")